# SpaceX Falcon 9 First Stage Landing Prediction

## Exploratory Data Analysis with SQL
This notebook explores SpaceX launch records using SQL queries on a local SQLite database.

## Contents
1. Set up SQL environment and load data
2. Run mission and payload analysis queries
3. Analyze landing outcomes and time-based patterns
4. Summarize key findings

## 1. Environment Setup and Data Loading
Initialize SQL support, connect to SQLite, and load the launch dataset.

In [1]:
# Load SQL extension for Jupyter
%load_ext sql

In [2]:
# Import required libraries
import csv
import sqlite3
import prettytable
import pandas as pd

prettytable.DEFAULT = 'DEFAULT'

# Create SQLite database connection
con = sqlite3.connect("../data/raw/my_data1.db")
cur = con.cursor()

In [3]:
# Connect SQL magic to the SQLite database
%sql sqlite:///../data/raw/my_data1.db

In [4]:
# Load the SpaceX launch data from CSV file
df = pd.read_csv('../data/raw/spacex_launch_data.csv')
df.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,6,2010-06-04,Falcon 9,6123.547647,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,8,2012-05-22,Falcon 9,525.000000,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,10,2013-03-01,Falcon 9,677.000000,ISS,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,11,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,12,2013-12-03,Falcon 9,3170.000000,GTO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


In [5]:
# Load the dataframe into the SQLite database table
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False)

90

## 2. Initial Data Exploration
Inspect table schema and distinct launch sites before running analytical queries.

In [6]:
# Display the column names and data types in the table
%sql PRAGMA table_info('SPACEXTBL')

 * sqlite:///../data/raw/my_data1.db
Done.


cid,name,type,notnull,dflt_value,pk
0,FlightNumber,INTEGER,0,None,0
1,Date,TEXT,0,None,0
2,BoosterVersion,TEXT,0,None,0
3,PayloadMass,REAL,0,None,0
4,Orbit,TEXT,0,None,0
5,LaunchSite,TEXT,0,None,0
6,Outcome,TEXT,0,None,0
7,Flights,INTEGER,0,None,0
8,GridFins,INTEGER,0,None,0
9,Reused,INTEGER,0,None,0


In [7]:
# Display the names of the unique launch sites in the space mission
%sql SELECT DISTINCT LaunchSite FROM SPACEXTBL

 * sqlite:///../data/raw/my_data1.db
Done.


LaunchSite
CCSFS SLC 40
VAFB SLC 4E
KSC LC 39A


### Query 1: Sample Records from CCSFS SLC 40

In [8]:
# Display 5 records where launch sites begin with 'CCSFS SLC 40'
%sql SELECT * FROM SPACEXTBL WHERE LaunchSite LIKE 'CCSFS SLC 40%' LIMIT 5

 * sqlite:///../data/raw/my_data1.db
Done.


FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
6,2010-06-04,Falcon 9,6123.547647058824,LEO,CCSFS SLC 40,None None,1,0,0,0,None,1.0,0,B0003,-80.577366,28.5618571
8,2012-05-22,Falcon 9,525.0,LEO,CCSFS SLC 40,None None,1,0,0,0,None,1.0,0,B0005,-80.577366,28.5618571
10,2013-03-01,Falcon 9,677.0,ISS,CCSFS SLC 40,None None,1,0,0,0,None,1.0,0,B0007,-80.577366,28.5618571
12,2013-12-03,Falcon 9,3170.0,GTO,CCSFS SLC 40,None None,1,0,0,0,None,1.0,0,B1004,-80.577366,28.5618571
13,2014-01-06,Falcon 9,3325.0,GTO,CCSFS SLC 40,None None,1,0,0,0,None,1.0,0,B1005,-80.577366,28.5618571


## 3. Payload Analysis
Use SQL aggregations to measure total and average payload metrics.

In [9]:
# Calculate the total payload mass carried by boosters launched to ISS (NASA CRS missions)
%sql SELECT SUM(PayloadMass) AS 'Total Payload Mass (kg)' FROM SPACEXTBL WHERE Orbit = 'ISS'

 * sqlite:///../data/raw/my_data1.db
Done.


Total Payload Mass (kg)
68878.7


### Query 3: Average Payload Mass for Falcon 9 Missions

In [10]:
# Calculate average payload mass carried by Falcon 9 boosters
%sql SELECT AVG(PayloadMass) AS 'Average Payload Mass (kg)' FROM SPACEXTBL WHERE BoosterVersion = 'Falcon 9'

 * sqlite:///../data/raw/my_data1.db
Done.


Average Payload Mass (kg)
6123.547647058824


## 4. Landing Outcome Analysis
Analyze successful and failed outcomes across mission conditions.

In [11]:
# Find the date when the first successful landing outcome on ground pad was achieved
%sql SELECT MIN(Date) AS 'First Successful Landing Date' FROM SPACEXTBL WHERE Outcome = 'True RTLS'

 * sqlite:///../data/raw/my_data1.db
Done.


First Successful Landing Date
2015-12-22


### Query 5: Successful ASDS Landings with Payload Between 4000 and 6000 kg

In [12]:
# List flight numbers with successful drone ship landings and payload mass between 4000-6000 kg
%sql SELECT FlightNumber FROM SPACEXTBL WHERE Outcome = 'True ASDS' AND PayloadMass BETWEEN 4000 AND 6000

 * sqlite:///../data/raw/my_data1.db
Done.


FlightNumber
29
33
38
49
67
71
85


### Query 6: Mission Outcome Counts

In [13]:
# List the total number of successful and failure mission outcomes
%sql SELECT Outcome, COUNT(Outcome) AS 'Count' FROM SPACEXTBL GROUP BY Outcome

 * sqlite:///../data/raw/my_data1.db
Done.


Outcome,Count
False ASDS,6
False Ocean,2
False RTLS,1
None ASDS,2
None None,19
True ASDS,41
True Ocean,5
True RTLS,14


### Query 7: Flights with Maximum Payload Mass

In [14]:
# List all flight numbers that carried the maximum payload mass (using subquery)
%sql SELECT FlightNumber FROM SPACEXTBL WHERE PayloadMass = (SELECT MAX(PayloadMass) FROM SPACEXTBL)

 * sqlite:///../data/raw/my_data1.db
Done.


FlightNumber
84
87
89
90
92
93
95
96
100
102


## 5. Time-Based Analysis
Evaluate mission outcomes by year and month to identify historical trends.

In [15]:
# List records showing month, outcome, flight number, and launch site for failed drone ship landings in 2015
%sql SELECT STRFTIME('%m', Date) AS 'Month', Outcome, FlightNumber, LaunchSite FROM SPACEXTBL WHERE Outcome = 'False ASDS' AND STRFTIME('%Y', Date) = '2015'

 * sqlite:///../data/raw/my_data1.db
Done.


Month,Outcome,FlightNumber,LaunchSite
01,False ASDS,19,CCSFS SLC 40
04,False ASDS,22,CCSFS SLC 40


### Query 9: Outcome Ranking from 2010-06-04 to 2017-03-20

In [16]:
# Rank the count of landing outcomes between 2010-06-04 and 2017-03-20 in descending order
%sql SELECT Outcome, COUNT(Outcome) AS 'Count' FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Outcome ORDER BY Count DESC

 * sqlite:///../data/raw/my_data1.db
Done.


Outcome,Count
None None,9
True ASDS,5
False ASDS,4
True RTLS,3
True Ocean,3
None ASDS,2
False Ocean,2


## Summary
This SQL EDA notebook answered key mission questions by querying the launch table for:

- Launch site coverage and sample missions
- Payload aggregates for ISS and Falcon 9 missions
- Ground and drone-ship landing outcomes
- Time-based outcome trends between 2010 and 2017

These query results provide a structured analytical baseline for the visualization and machine learning notebooks.